# Python Async/Await Explained

This notebook focuses on **execution semantics**, because most async bugs come from misunderstanding *when* code runs, *what* gets scheduled, and *what* blocks the event loop.

## Mental Model

Keep these rules in mind:

- `async def` defines a coroutine function. Calling it returns a **coroutine object**.
- `await` pauses the current coroutine and gives control back to the event loop.
- `asyncio.create_task(...)` schedules work to run concurrently.
- Blocking code like `time.sleep(...)` freezes the event loop.
- Async gives **concurrency**, not CPU parallelism.

In [4]:
import asyncio
import time

print('asyncio imported successfully')

asyncio imported successfully


## 1. Calling an async function does not run it

This is the most common misunderstanding. A coroutine is lazy until it is awaited or scheduled.

In [5]:
async def fetch_value():
    print('fetch_value started')
    await asyncio.sleep(0.2)
    print('fetch_value finished')
    return 42

coro = fetch_value()
print('Object returned:', coro)
print('Type:', type(coro))

# Close the demo coroutine so the notebook does not emit an
# un-awaited coroutine warning after showing the object.
coro.close()

Object returned: <coroutine object fetch_value at 0x000002A5B6301C00>
Type: <class 'coroutine'>


The previous cell created a coroutine object, but nothing inside `fetch_value` executed yet.

In [6]:
result = await fetch_value()
print('Awaited result:', result)

fetch_value started
fetch_value finished
Awaited result: 42


## 2. `await` only works inside async code or notebook cells with top-level await

In a normal `.py` file, `await` is only valid inside `async def`. In Jupyter notebooks, top-level `await` is usually supported, which is why the previous cell works here.

From regular synchronous Python code, the entry point is usually `asyncio.run(...)`.

In [17]:
async def main_from_sync_context():
    value = await fetch_value()
    return f'value from async entry point: {value}'

try:
    asyncio.get_running_loop()
except RuntimeError:
    print(asyncio.run(main_from_sync_context()))
else:
    print('Notebook event loop detected; using top-level await instead of asyncio.run(...)')
    print(await main_from_sync_context())

Notebook event loop detected; using top-level await instead of asyncio.run(...)
fetch_value started
fetch_value finished
value from async entry point: 42


If you run the previous cell in a notebook that already has an active event loop, `asyncio.run(...)` may raise `RuntimeError: asyncio.run() cannot be called from a running event loop`. In notebooks, prefer top-level `await` instead.

## 3. Blocking the event loop vs yielding control

The easiest way to kill async performance is to block the loop with synchronous work.

In [18]:
async def bad_sleep(name):
    start = time.perf_counter()
    print(f'{name} starting blocking sleep')
    time.sleep(1)
    elapsed = time.perf_counter() - start
    print(f'{name} finished after {elapsed:.2f}s')

async def demo_blocking():
    start = time.perf_counter()
    await asyncio.gather(bad_sleep('A'), bad_sleep('B'))
    print(f'total blocking time: {time.perf_counter() - start:.2f}s')

await demo_blocking()

A starting blocking sleep
A finished after 1.00s
B starting blocking sleep
B finished after 1.00s
total blocking time: 2.00s


In [19]:
async def good_sleep(name):
    start = time.perf_counter()
    print(f'{name} starting non-blocking sleep')
    await asyncio.sleep(1)
    elapsed = time.perf_counter() - start
    print(f'{name} finished after {elapsed:.2f}s')

async def demo_non_blocking():
    start = time.perf_counter()
    await asyncio.gather(good_sleep('A'), good_sleep('B'))
    print(f'total non-blocking time: {time.perf_counter() - start:.2f}s')

await demo_non_blocking()

A starting non-blocking sleep
B starting non-blocking sleep
A finished after 1.01s
B finished after 1.01s
total non-blocking time: 1.01s


With blocking sleep, both tasks effectively run one after another. With `await asyncio.sleep(...)`, both tasks make progress concurrently.

## 4. Sequential awaits vs real concurrency

Code can look asynchronous but still run sequentially.

In [10]:
async def fetch_named(name, delay):
    await asyncio.sleep(delay)
    return f'{name} done'

async def run_sequential():
    start = time.perf_counter()
    first = await fetch_named('first', 1)
    second = await fetch_named('second', 1)
    elapsed = time.perf_counter() - start
    return first, second, elapsed

async def run_concurrent():
    start = time.perf_counter()
    first, second = await asyncio.gather(
        fetch_named('first', 1),
        fetch_named('second', 1),
    )
    elapsed = time.perf_counter() - start
    return first, second, elapsed

sequential_result = await run_sequential()
concurrent_result = await run_concurrent()

print('sequential:', sequential_result)
print('concurrent:', concurrent_result)

sequential: ('first done', 'second done', 2.0164325999994617)
concurrent: ('first done', 'second done', 1.0139828999999736)


## 5. Coroutine vs task

A coroutine object is just a plan for work. A task is scheduled work that starts running under the event loop.

In [11]:
async def worker(name, delay):
    print(f'{name} started')
    await asyncio.sleep(delay)
    print(f'{name} finished')
    return name

coro = worker('coroutine-object', 0.3)
task = asyncio.create_task(worker('scheduled-task', 0.3))

print('Coroutine:', coro)
print('Task:', task)

await asyncio.sleep(0.1)
print('After 0.1s, task already had a chance to run.')
await task
await coro

Coroutine: <coroutine object worker at 0x000002A5B6494AC0>
Task: <Task pending name='Task-74' coro=<worker() running at C:\Users\weijiexia\AppData\Local\Temp\ipykernel_25136\181655200.py:1>>
scheduled-task started
After 0.1s, task already had a chance to run.
scheduled-task finished
coroutine-object started
coroutine-object finished


'coroutine-object'

Notice the timing difference: the task starts once scheduled, while the unscheduled coroutine waits until it is awaited.

## 6. Why fire-and-forget is often a bug

If you create a task and never observe it, exceptions may be missed and shutdown behavior becomes fragile.

In [12]:
async def failing_task():
    await asyncio.sleep(0.2)
    raise ValueError('background task failed')

task = asyncio.create_task(failing_task())

try:
    await task
except ValueError as exc:
    print('Caught task error:', exc)

Caught task error: background task failed


A managed task is much safer than `asyncio.create_task(...)` with no reference and no error handling.

## 7. Cancellation is part of normal async control flow

Cancellation is not an unusual edge case. Any long-lived task should assume it may be cancelled.

In [13]:
async def cancellable_worker():
    try:
        print('worker running')
        await asyncio.sleep(5)
    except asyncio.CancelledError:
        print('cleanup before cancellation')
        raise

task = asyncio.create_task(cancellable_worker())
await asyncio.sleep(0.2)
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print('caller observed cancellation')

worker running
cleanup before cancellation
caller observed cancellation


## 8. Async comprehensions and iteration

Two easy mistakes are building a list of coroutine objects by accident and using regular `for` with an async generator.

In [14]:
items = [1, 2, 3]
coroutines = [fetch_named(f'item-{item}', 0.1) for item in items]
print('List contents:', coroutines)

for coroutine in coroutines:
    coroutine.close()

results = await asyncio.gather(*(fetch_named(f'item-{item}', 0.1) for item in items))
print('Actual results:', results)

List contents: [<coroutine object fetch_named at 0x000002A5B6476330>, <coroutine object fetch_named at 0x000002A5B6476E90>, <coroutine object fetch_named at 0x000002A5B6476DC0>]
Actual results: ['item-1 done', 'item-2 done', 'item-3 done']


In [15]:
async def async_counter(limit):
    for number in range(limit):
        await asyncio.sleep(0.1)
        yield number

values = []
async for value in async_counter(3):
    values.append(value)

print(values)

[0, 1, 2]


## 9. Async is not for pure CPU work

If the work is CPU-bound and does not `await`, putting it in `async def` does not make it cooperative. It still blocks when executed. For blocking code, offload it with `asyncio.to_thread(...)` or use multiprocessing for real CPU parallelism.

In [16]:
def blocking_io_like_work():
    time.sleep(1)
    return 'finished in worker thread'

thread_result = await asyncio.to_thread(blocking_io_like_work)
print(thread_result)

finished in worker thread
